# Lab 04 — Modelado Supervisado (demo en Colab)

Analitica de Datos · Pipeline de Olist

Este notebook clona el repo, instala dependencias, corre los 54 tests y reproduce en vivo el entrenamiento/evaluacion de los pipelines de regresion (`tiempo_entrega_dias`) y clasificacion (`review_negativa`) con datos reales de Olist. Pensado para correr desde cualquier navegador, sin instalar nada localmente.

Repo: https://github.com/Thebird45/analytics-project

## 1. Clonar el repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/Thebird45/analytics-project.git
%cd analytics-project

In [ ]:
!pip install -q -e .

## 2. Datos de Olist

`data/raw/` no viene en el repo (esta en `.gitignore`, son datasets pesados). La celda de abajo descarga los 9 CSV directamente desde el folder de Google Drive compartido — no requiere montar tu cuenta de Drive ni subir nada a mano. Si el link deja de estar disponible, usa la Opcion B (subida manual) como respaldo.

In [ ]:
# --- OPCION A (recomendada): descargar el folder de Drive compartido ---
!pip install -q gdown

import glob
import os
import shutil

import gdown

FOLDER_URL = 'https://drive.google.com/drive/folders/11ExZHMTMb8hNkdb3SwjeryD2kGmRpHng'

os.makedirs('data/raw', exist_ok=True)
shutil.rmtree('_drive_download', ignore_errors=True)
gdown.download_folder(url=FOLDER_URL, output='_drive_download', quiet=False, use_cookies=False)

for ruta_csv in glob.glob('_drive_download/**/*.csv', recursive=True):
    shutil.copy(ruta_csv, 'data/raw/')
shutil.rmtree('_drive_download', ignore_errors=True)

print(sorted(os.listdir('data/raw')))

In [ ]:
# --- OPCION B: subida manual (respaldo si la Opcion A falla) ---
# Descomenta y corre esta celda si el folder de Drive no esta disponible.

# from google.colab import files
# import os
#
# os.makedirs('data/raw', exist_ok=True)
# subidos = files.upload()  # selecciona los 9 CSV de Olist en el dialogo
# for nombre in subidos:
#     os.rename(nombre, f'data/raw/{nombre}')
# print(sorted(os.listdir('data/raw')))

## 3. Verificacion: tests + calidad de codigo

In [ ]:
!python -m pytest -v

In [ ]:
!pip install -q ruff
!ruff check .
!ruff format --check .

## 4. Entrenamiento y evaluacion con datos reales

De aqui en adelante se reproduce, celda por celda, lo que hace `scripts/verificacion_manual_lab04.py` — util para explicar cada paso durante la sustentacion.

In [ ]:
import sys

sys.path.insert(0, 'src')  # por si 'pip install -e .' no corrio en este runtime

from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score

from analytics import extract, transform, train, evaluate

tablas = extract.cargar_olist('data/raw')

### 4.1 Regresion — `tiempo_entrega_dias`

In [ ]:
df = extract.construir_dataset_base(tablas)
df = transform.agregar_features_fecha(df)
df = transform.agregar_encoding_estado(df)
df = transform.agregar_ticket_historico(df)

flete_agg = tablas['items'].groupby('order_id', as_index=False).agg(
    flete_total=('freight_value', 'sum')
)
df = df.merge(flete_agg, on='order_id', how='left')

features_reg = [
    'n_items', 'precio_total', 'flete_total', 'customer_state_freq',
    'dia_semana_compra', 'mes_compra', 'ticket_promedio_historico',
]
X = df[features_reg]
y = df['tiempo_entrega_dias']
mask = y.notna()
X, y = X[mask], y[mask]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

pipeline_reg = train.entrenar_pipeline_regresion(X_train, y_train)
metricas_val = evaluate.evaluar_regresion(y_val, pipeline_reg.predict(X_val))
metricas_test = evaluate.evaluar_regresion(y_test, pipeline_reg.predict(X_test))

print('Metricas validation:', metricas_val)
print('Metricas test:      ', metricas_test)

train.guardar_pipeline(pipeline_reg, 'models/pipeline_tiempo_entrega.joblib')

### 4.2 Clasificacion — `review_negativa`

In [ ]:
reviews_dedup = (
    tablas['reviews']
    .sort_values('review_creation_date')
    .drop_duplicates('order_id', keep='last')[['order_id', 'review_score']]
)
df_clf = df.merge(reviews_dedup, on='order_id', how='inner')
df_clf['review_negativa'] = df_clf['review_score'] <= 2

print(df_clf['review_negativa'].value_counts(normalize=True))

features_clf = [
    'n_items', 'precio_total', 'flete_total', 'customer_state_freq',
    'tiempo_entrega_dias', 'entregado_tarde', 'ticket_promedio_historico',
]
Xc = df_clf[features_clf]
yc = df_clf['review_negativa']

Xc_train, Xc_eval, yc_train, yc_eval = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc
)

pipeline_clf = train.entrenar_pipeline_clasificacion(Xc_train, yc_train)
y_pred = pipeline_clf.predict(Xc_eval)
y_proba = pipeline_clf.predict_proba(Xc_eval)[:, 1]

metricas_clf = evaluate.evaluar_clasificacion(yc_eval, y_pred, y_proba)
comparacion = evaluate.comparar_con_baseline(yc_train, yc_eval, y_pred)

print('Metricas del modelo:', metricas_clf)
print('Comparacion vs. baseline:', comparacion)

train.guardar_pipeline(pipeline_clf, 'models/pipeline_review_negativa.joblib')

### 4.3 Pregunta 9.3 — efecto de `class_weight="balanced"` sobre el recall

In [ ]:
pipeline_sin_balanceo = train.entrenar_pipeline_clasificacion(Xc_train, yc_train)
pipeline_sin_balanceo.named_steps['modelo'].set_params(class_weight=None)
pipeline_sin_balanceo.fit(Xc_train, yc_train)

recall_balanced = recall_score(yc_eval, y_pred)
recall_sin_balanceo = recall_score(yc_eval, pipeline_sin_balanceo.predict(Xc_eval))

print(f"Recall con class_weight='balanced': {recall_balanced:.3f}")
print(f"Recall con class_weight=None:       {recall_sin_balanceo:.3f}")

## 5. Resumen de la corrida

In [ ]:
print('=== RESUMEN ===')
print(f"9.1 mejora accuracy vs baseline: {comparacion['mejora']:.4f}")
print(f"9.1 accuracy_baseline={comparacion['accuracy_baseline']:.3f} "
      f"accuracy_modelo={comparacion['accuracy_modelo']:.3f}")
print(f"9.1 f1={metricas_clf['f1']:.3f} roc_auc={metricas_clf['roc_auc']:.3f}")
print(f"9.2 rmse_val={metricas_val['rmse']:.3f} rmse_test={metricas_test['rmse']:.3f}")
print(f"9.3 recall_balanced={recall_balanced:.3f} recall_sin_balanceo={recall_sin_balanceo:.3f}")